# Two-Doer Bottleneck: Evaluation Notebook

Downloads the trained checkpoint from HuggingFace and runs:
1. **Fast batched rollout** (1000 episodes) — empirical message→pick and message→nav correlations
2. **Controlled probes** — single-message nav and pick semantics
3. **GIF visualization** of greedy episodes
4. **Bit-level compositionality** analysis

## 1. Install Dependencies

In [ ]:
!pip install -q 'jax[cuda12]' flax optax chex pillow numpy huggingface_hub navix


## 2. Imports

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter
from itertools import product

import numpy as np
import jax
import jax.numpy as jnp
from PIL import Image
from IPython.display import display, Image as IPImage

# Ensure repo root is on path
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.seer import Seer
from models.doer import Doer
from envs.two_doer_grid import TwoDoerBottleneckEnv
from training.action_masking import mask_pick_actions_until_menu_visible
from training.message_masking import hard_mask_inactive_message_bits

print(f"JAX backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")


## 3. Configuration

In [ ]:
HF_REPO      = "jonny-vr/two-doers-one-seer-curriculum"  # Default pretrained model
FSQ_LEVELS   = [2, 2, 2, 2]
HIDDEN_SIZE  = 128
NUM_EPISODES = 1000   # Episodes for empirical correlation analysis
NUM_ENVS     = 128    # Parallel envs for fast rollout
SEED         = 42

# Environment settings (must match training)
ENV_CFG = dict(
    grid_height=10, grid_width=12, local_view_size=3, corridor_length=3,
    max_steps=48, doer_perception_level=2,
    pick_object_max_steps=8, pick_object_listen_steps=1,
)

DOER_KEYS   = ["doer_a", "doer_b"]
COLOR_NAMES = ["red", "yellow", "green", "blue"]
SHAPE_NAMES = ["plus", "x", "solid_square", "frame"]

def item_label(item_id):
    if item_id is None or item_id < 0:
        return None
    return f"{COLOR_NAMES[item_id // 4]}_{SHAPE_NAMES[item_id % 4]}"

def action_label(action_id):
    labels = ["stay", "up", "down", "left", "right", "pick_0", "pick_1", "pick_2", "pick_3"]
    return labels[action_id] if action_id < len(labels) else str(action_id)

def all_messages(fsq_levels):
    return list(product(*[range(l) for l in fsq_levels]))

print("Config ready.")


## 4. Download Checkpoint from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

print(f"Downloading checkpoint from {HF_REPO} ...")
local_dir = snapshot_download(repo_id=HF_REPO)
checkpoint_path = Path(local_dir)
print(f"Downloaded to: {checkpoint_path}")

# Find the checkpoint directory
def find_checkpoint(base: Path) -> Path:
    # Direct orbax checkpoint
    if (base / "_CHECKPOINT_METADATA").exists():
        return base
    # checkpoint_N wrapper directories
    candidates = sorted(
        [d for d in base.iterdir() if d.is_dir() and d.name.startswith("checkpoint_")],
        key=lambda d: int(d.name.split("_")[-1]) if d.name.split("_")[-1].isdigit() else -1,
    )
    if candidates:
        return candidates[-1]  # latest
    return base

resolved = find_checkpoint(checkpoint_path)
print(f"Using checkpoint: {resolved}")


In [ ]:
import orbax.checkpoint as ocp

checkpointer = ocp.PyTreeCheckpointer()
params = checkpointer.restore(str(resolved))
print("Checkpoint loaded. Keys:", list(params.keys()))


## 5. Build Environment and Models

In [ ]:
def build_env(selection_phase_level=2, **overrides):
    cfg = {**ENV_CFG, **overrides}
    return TwoDoerBottleneckEnv(
        selection_phase_level=selection_phase_level,
        goal_reward=2.0,
        wrong_selection_penalty=0.15,
        wrong_selection_penalty_after_first=0.30,
        step_penalty=0.03,
        wall_penalty=0.02,
        collision_penalty=0.05,
        progress_reward_scale=0.1,
        max_selection_attempts=4,
        **cfg,
    )

env  = build_env(selection_phase_level=2)
seer = Seer(fsq_levels=FSQ_LEVELS, num_actions=env.num_actions, num_message_heads=env.num_doers)
doer = Doer(fsq_levels=FSQ_LEVELS, num_actions=env.num_actions)

print(f"Env: {env.phase_name} | Actions: {env.num_actions} | Doers: {env.num_doers}")


## 6. Fast Batched Rollout (1000 Episodes)

Uses `jax.lax.scan` with 128 parallel environments — compiles once, runs fast.

In [ ]:
def run_fast_rollout(env, params, seer, doer, hidden_size, num_episodes, rng, num_envs=128):
    """Batched rollout via lax.scan. Returns raw per-step records for all envs."""
    num_doers    = env.num_doers
    max_steps    = env.phase_max_steps
    steps_per_env = ((num_episodes + num_envs - 1) // num_envs + 1) * max_steps
    active_bits  = env.active_message_bits

    rng, reset_rng = jax.random.split(rng)
    obs0, state0 = env.reset_batch(jax.random.split(reset_rng, num_envs))

    init_seer_carry = seer.initialize_carry(batch_size=num_envs, hidden_size=hidden_size)
    init_doer_carry = jax.tree_util.tree_map(
        lambda x: x.reshape((num_envs, num_doers, hidden_size)),
        doer.initialize_carry(batch_size=num_envs * num_doers, hidden_size=hidden_size),
    )

    def scan_step(carry, step_rng):
        obs, state, seer_carry, doer_carry = carry

        new_seer_carry, messages, _, _ = seer.apply(
            {"params": params["seer"]}, seer_carry,
            obs["global_map"], obs["symbolic_state"], obs["target_images"],
        )
        messages = hard_mask_inactive_message_bits(messages, active_bits=active_bits)

        flat_lv   = obs["local_views"].reshape((num_envs * num_doers,) + obs["local_views"].shape[2:])
        flat_prop = obs["proprioceptions"].reshape((num_envs * num_doers,) + obs["proprioceptions"].shape[2:])
        flat_msg  = messages.reshape((num_envs * num_doers,) + messages.shape[2:])
        flat_menu = obs["menu_images"].reshape((num_envs * num_doers,) + obs["menu_images"].shape[2:])
        flat_dc   = jax.tree_util.tree_map(lambda x: x.reshape((num_envs * num_doers, hidden_size)), doer_carry)

        new_flat_dc, flat_logits = doer.apply({"params": params["doer"]}, flat_dc, flat_lv, flat_prop, flat_msg, flat_menu)
        logits_2d = flat_logits.reshape((num_envs, num_doers, flat_logits.shape[-1]))
        masked    = mask_pick_actions_until_menu_visible(
            logits_2d, obs["menu_images"], pick_only_phase=env.is_pick_object_phase, pick_available=obs["pick_available"]
        )
        actions = jnp.argmax(masked, axis=-1).astype(jnp.int32)  # (num_envs, num_doers)

        # First pick per episode
        is_pick       = actions >= 5
        is_first_pick = jnp.logical_and(is_pick, ~state.has_selected)
        pick_slot     = jnp.clip(actions - 5, 0, 3)
        env_idx       = jnp.arange(num_envs)[:, None]
        doer_idx      = jnp.arange(num_doers)[None, :]
        chosen_items  = state.shuffled_menus[env_idx, doer_idx, pick_slot]
        chosen_items  = jnp.where(is_first_pick, chosen_items, jnp.full_like(chosen_items, -1))

        # Nav actions (movement only)
        is_nav    = actions < 5
        nav_actions = jnp.where(is_nav, actions, jnp.full_like(actions, -1))

        step_keys = jax.random.split(step_rng, num_envs)
        next_obs, next_state, _, done, _ = env.step_batch(step_keys, state, actions)

        new_seer_carry = jax.tree_util.tree_map(
            lambda new, zero: jnp.where(done[:, None], zero, new), new_seer_carry, init_seer_carry
        )
        new_doer_carry = jax.tree_util.tree_map(
            lambda x: x.reshape((num_envs, num_doers, hidden_size)), new_flat_dc
        )
        new_doer_carry = jax.tree_util.tree_map(
            lambda new, zero: jnp.where(done[:, None, None], zero, new), new_doer_carry, init_doer_carry
        )

        record = {
            "messages":     messages,              # (E, D, msg_dim)
            "chosen_items": chosen_items,          # (E, D)  -1=no first-pick
            "target_items": state.target_items,    # (E, D)
            "nav_actions":  nav_actions,           # (E, D)  -1=pick step
        }
        return (next_obs, next_state, new_seer_carry, new_doer_carry), record

    print(f"  Compiling rollout ({num_envs} envs x {steps_per_env} steps)...")
    _, records = jax.lax.scan(scan_step, (obs0, state0, init_seer_carry, init_doer_carry),
                               jax.random.split(rng, steps_per_env))
    jax.block_until_ready(records)
    n_picks = int(jnp.sum(records["chosen_items"] >= 0))
    n_nav   = int(jnp.sum(records["nav_actions"] >= 0))
    print(f"  Done. {n_picks} pick events, {n_nav} nav-action steps.")
    return records


rng = jax.random.PRNGKey(SEED)
rng, rollout_rng = jax.random.split(rng)
records = run_fast_rollout(env, params, seer, doer, HIDDEN_SIZE, NUM_EPISODES, rollout_rng, num_envs=NUM_ENVS)


## 7. Empirical Message → Pick Correlations

In [ ]:
def analyze_pick_correlations(records):
    messages     = np.asarray(records["messages"])    # (T, E, D, msg_dim)
    chosen_items = np.asarray(records["chosen_items"]) # (T, E, D)
    target_items = np.asarray(records["target_items"]) # (T, E, D)

    results = {}
    for di, dk in enumerate(DOER_KEYS):
        msgs_flat   = messages[:, :, di, :].reshape(-1, messages.shape[-1])
        chosen_flat = chosen_items[:, :, di].reshape(-1)
        target_flat = target_items[:, :, di].reshape(-1)
        pick_mask   = chosen_flat >= 0

        buckets = {}
        for i in np.where(pick_mask)[0]:
            msg = tuple(int(x) for x in msgs_flat[i])
            ci, ti = int(chosen_flat[i]), int(target_flat[i])
            if msg not in buckets:
                buckets[msg] = {"item_counts": Counter(), "color_counts": Counter(),
                                "shape_counts": Counter(), "correct": 0, "total": 0}
            b = buckets[msg]
            b["total"] += 1
            b["item_counts"][ci] += 1
            b["color_counts"][COLOR_NAMES[ci // 4]] += 1
            b["shape_counts"][SHAPE_NAMES[ci % 4]] += 1
            if ci == ti:
                b["correct"] += 1

        summary = {}
        for msg, b in sorted(buckets.items(), key=lambda x: -x[1]["total"]):
            n = b["total"]
            top_item_id, top_item_n = b["item_counts"].most_common(1)[0]
            top_color,   top_color_n = b["color_counts"].most_common(1)[0]
            top_shape,   top_shape_n = b["shape_counts"].most_common(1)[0]
            summary[str(msg)] = {
                "message": list(msg), "n_picks": n,
                "correct_rate": b["correct"] / n,
                "top_item": item_label(top_item_id), "top_item_rate": top_item_n / n,
                "top_color": top_color,              "top_color_rate": top_color_n / n,
                "top_shape": top_shape,              "top_shape_rate": top_shape_n / n,
            }
        results[dk] = summary
    return results


pick_correlations = analyze_pick_correlations(records)

for dk in DOER_KEYS:
    print(f"\n{'='*80}")
    print(f"  {dk.upper()} — Message->Pick Correlations")
    print(f"{'='*80}")
    hdr = f"{'Message':<22} {'N':>5}  {'Top Item':<26} {'Top Color':<18} {'Top Shape':<18} {'Correct%':>8}"
    print(hdr)
    print("-" * len(hdr))
    for e in pick_correlations[dk].values():
        print(f"{str(e['message']):<22} {e['n_picks']:>5}  "
              f"{e['top_item']+' ('+f"{e['top_item_rate']:.2f}" + ')':  <26} "
              f"{e['top_color']+' ('+f"{e['top_color_rate']:.2f}" + ')':  <18} "
              f"{e['top_shape']+' ('+f"{e['top_shape_rate']:.2f}" + ')':  <18} "
              f"{e['correct_rate']:>8.2f}")


## 8. Empirical Message → Nav-Action Correlations

In [ ]:
def analyze_nav_correlations(records):
    messages    = np.asarray(records["messages"])    # (T, E, D, msg_dim)
    nav_actions = np.asarray(records["nav_actions"]) # (T, E, D)  -1=pick step

    results = {}
    for di, dk in enumerate(DOER_KEYS):
        msgs_flat = messages[:, :, di, :].reshape(-1, messages.shape[-1])
        nav_flat  = nav_actions[:, :, di].reshape(-1)
        nav_mask  = nav_flat >= 0

        buckets = {}
        for i in np.where(nav_mask)[0]:
            msg = tuple(int(x) for x in msgs_flat[i])
            act = int(nav_flat[i])
            if msg not in buckets:
                buckets[msg] = {"action_counts": Counter(), "total": 0}
            buckets[msg]["action_counts"][act] += 1
            buckets[msg]["total"] += 1

        summary = {}
        for msg, b in sorted(buckets.items(), key=lambda x: -x[1]["total"]):
            n = b["total"]
            top_act, top_n = b["action_counts"].most_common(1)[0]
            summary[str(msg)] = {
                "message": list(msg), "n_steps": n,
                "top_action": action_label(top_act),
                "action_purity": top_n / n,
                "action_counts": {action_label(a): c for a, c in b["action_counts"].items()},
            }
        results[dk] = summary
    return results


nav_correlations = analyze_nav_correlations(records)

for dk in DOER_KEYS:
    print(f"\n{'='*70}")
    print(f"  {dk.upper()} — Message->Nav-Action Correlations")
    print(f"{'='*70}")
    hdr = f"{'Message':<22} {'N':>7}  {'Top Action':<12} {'Purity':>7}  Action distribution"
    print(hdr)
    print("-" * len(hdr))
    for e in nav_correlations[dk].values():
        dist = "  ".join(f"{a}:{c}" for a, c in sorted(e["action_counts"].items(), key=lambda x: -x[1]))
        print(f"{str(e['message']):<22} {e['n_steps']:>7}  {e['top_action']:<12} {e['action_purity']:>7.2f}  {dist}")


## 9. Bit-Level Compositionality

In [ ]:
def analyze_bit_compositionality(pick_correlations, fsq_levels):
    results = {}
    for dk in DOER_KEYS:
        bit_results = {}
        for bit_idx in range(len(fsq_levels)):
            groups = {v: {"color_counts": Counter(), "shape_counts": Counter(), "n": 0}
                      for v in range(fsq_levels[bit_idx])}
            for entry in pick_correlations[dk].values():
                v = entry["message"][bit_idx]
                n = entry["n_picks"]
                # weight by pick count
                groups[v]["color_counts"][entry["top_color"]] += int(n * entry["top_color_rate"])
                groups[v]["shape_counts"][entry["top_shape"]] += int(n * entry["top_shape_rate"])
                groups[v]["n"] += n
            bit_summary = {}
            for v, g in groups.items():
                if g["n"] == 0:
                    bit_summary[v] = None
                    continue
                top_color = g["color_counts"].most_common(1)
                top_shape = g["shape_counts"].most_common(1)
                n = g["n"]
                bit_summary[v] = {
                    "n": n,
                    "top_color":    top_color[0][0] if top_color else "?",
                    "color_purity": top_color[0][1] / n if top_color else 0.0,
                    "top_shape":    top_shape[0][0] if top_shape else "?",
                    "shape_purity": top_shape[0][1] / n if top_shape else 0.0,
                }
            bit_results[bit_idx] = bit_summary
        results[dk] = bit_results
    return results


bit_compositionality = analyze_bit_compositionality(pick_correlations, FSQ_LEVELS)

for dk in DOER_KEYS:
    print(f"\n{dk.upper()} — Bit-level compositionality:")
    for bit_idx, bit_summary in bit_compositionality[dk].items():
        parts = []
        for v, g in sorted(bit_summary.items()):
            if g is None:
                continue
            parts.append(f"bit{bit_idx}={v}: n={g['n']} | color->{g['top_color']} ({g['color_purity']:.2f}) | shape->{g['top_shape']} ({g['shape_purity']:.2f})")
        print("  " + "  //  ".join(parts))


## 10. Success Rate

In [ ]:
chosen = np.asarray(records["chosen_items"])
target = np.asarray(records["target_items"])
pick_mask    = chosen >= 0
success_rate = float(np.sum((chosen == target) & pick_mask) / max(1, np.sum(pick_mask)))
print(f"Pick success rate (first pick): {success_rate:.3f}")


## 11. Controlled Nav Probe

For each message, reset the Doer at the corridor-entry position and record which navigation action it takes. Repeated across all candidate row positions to compute consistency.

In [ ]:
nav_env = build_env(selection_phase_level=2)
candidate_rows = np.asarray(nav_env._candidate_rows)
messages_list  = all_messages(FSQ_LEVELS)
all_seqs_array = jnp.asarray([[list(m)] for m in messages_list], dtype=jnp.float32)  # (N, 1, msg_dim)
zero_menu      = jnp.zeros((1, 4, 5, 5, 3), dtype=jnp.float32)

per_msg_actions = {dk: {str(list(m)): [] for m in messages_list} for dk in DOER_KEYS}

for row in candidate_rows:
    fixed_pos = jnp.asarray([[int(row), nav_env._left_col], [int(row), nav_env._right_col]], dtype=jnp.int32)
    nav_obs, _ = nav_env.reset(jax.random.PRNGKey(0), fixed_positions=fixed_pos)

    for di, dk in enumerate(DOER_KEYS):
        local_obs  = nav_obs["local_views"][di][None, ...]
        propr      = nav_obs["proprioceptions"][di][None, ...]
        carry      = doer.initialize_carry(batch_size=len(messages_list), hidden_size=HIDDEN_SIZE)

        # Broadcast obs to match batch size
        N = len(messages_list)
        local_b = jnp.broadcast_to(local_obs, (N,) + local_obs.shape[1:])
        prop_b  = jnp.broadcast_to(propr,     (N,) + propr.shape[1:])
        menu_b  = jnp.broadcast_to(zero_menu, (N,) + zero_menu.shape[1:])
        msgs_b  = all_seqs_array[:, 0, :]  # (N, msg_dim)

        _, logits = doer.apply({"params": params["doer"]}, carry, local_b, prop_b, msgs_b, menu_b)
        # Mask picks (not in nav phase)
        logits = logits.at[:, 5:].set(-1e9)
        actions = np.asarray(jnp.argmax(logits, axis=-1))

        for i, msg in enumerate(messages_list):
            per_msg_actions[dk][str(list(msg))].append(int(actions[i]))

# Compute modal action and consistency
nav_probe_results = {}
for dk in DOER_KEYS:
    nav_probe_results[dk] = {}
    for msg_key, acts in per_msg_actions[dk].items():
        counter = Counter(acts)
        modal_act, modal_n = counter.most_common(1)[0]
        nav_probe_results[dk][msg_key] = {
            "modal_action": action_label(modal_act),
            "consistency":  modal_n / len(acts),
            "action_counts": {action_label(a): c for a, c in counter.items()},
        }

print("Nav probe results (modal action / consistency across positions):")
for dk in DOER_KEYS:
    print(f"\n{dk.upper()}:")
    hdr = f"  {'Message':<22} {'Modal action':<14} {'Consistency':>12}  Counts"
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for msg_key, r in nav_probe_results[dk].items():
        counts = "  ".join(f"{a}:{c}" for a, c in sorted(r["action_counts"].items(), key=lambda x: -x[1]))
        print(f"  {msg_key:<22} {r['modal_action']:<14} {r['consistency']:>12.2f}  {counts}")


## 12. GIF Visualization

In [ ]:
NUM_GIFS = 3  # How many episodes to visualize

def run_greedy_episode(env, params, seer, doer, rng, hidden_size, max_steps):
    """Run a single greedy episode and return rendered frames."""
    rng, reset_rng = jax.random.split(rng)
    obs, state = env.reset(reset_rng)

    seer_carry = seer.initialize_carry(batch_size=1, hidden_size=hidden_size)
    doer_carry = jax.tree_util.tree_map(
        lambda x: x.reshape((1, env.num_doers, hidden_size)),
        doer.initialize_carry(batch_size=env.num_doers, hidden_size=hidden_size),
    )

    frames = [env.render(state)]
    done, step_count, success = False, 0, False

    while not done and step_count < max_steps:
        seer_carry, messages, _, _ = seer.apply(
            {"params": params["seer"]}, seer_carry,
            obs["global_map"][None], obs["symbolic_state"][None], obs["target_images"][None],
        )
        messages = hard_mask_inactive_message_bits(messages, active_bits=env.active_message_bits)

        flat_lv   = obs["local_views"][None].reshape((env.num_doers,) + obs["local_views"].shape[1:])
        flat_prop = obs["proprioceptions"][None].reshape((env.num_doers,) + obs["proprioceptions"].shape[1:])
        flat_msg  = messages.reshape((env.num_doers,) + messages.shape[2:])
        flat_menu = obs["menu_images"][None].reshape((env.num_doers,) + obs["menu_images"].shape[1:])
        flat_dc   = jax.tree_util.tree_map(lambda x: x.reshape((env.num_doers, hidden_size)), doer_carry)

        new_flat_dc, flat_logits = doer.apply({"params": params["doer"]}, flat_dc, flat_lv, flat_prop, flat_msg, flat_menu)
        logits_2d = flat_logits[None]  # (1, num_doers, num_actions)
        masked    = mask_pick_actions_until_menu_visible(
            logits_2d, obs["menu_images"][None], pick_only_phase=env.is_pick_object_phase,
            pick_available=obs["pick_available"][None],
        )
        actions   = jnp.argmax(masked[0], axis=-1)  # (num_doers,)
        doer_carry = jax.tree_util.tree_map(lambda x: x.reshape((1, env.num_doers, hidden_size)), new_flat_dc)

        rng, step_rng = jax.random.split(rng)
        obs, state, _, done, info = env.step(step_rng, state, actions)
        frames.append(env.render(state))
        step_count += 1
        if done:
            success = bool(info.get("task_success", False))

    return frames, success


Path("gifs").mkdir(exist_ok=True)
for ep_idx in range(NUM_GIFS):
    rng, ep_rng = jax.random.split(rng)
    frames, success = run_greedy_episode(env, params, seer, doer, ep_rng, HIDDEN_SIZE, max_steps=env.phase_max_steps)
    gif_path = f"gifs/episode_{ep_idx:03d}.gif"
    pil_frames = [Image.fromarray(f) for f in frames]
    pil_frames[0].save(gif_path, save_all=True, append_images=pil_frames[1:], loop=0, duration=200)
    print(f"Episode {ep_idx}: {'SUCCESS' if success else 'FAIL'} ({len(frames)} steps) -> {gif_path}")
    display(IPImage(filename=gif_path))
